# HTB Spam Classification: Naive Bayes by hand and in Python

Module 292, Section 9. This notebook reproduces **only the course's supplied two-feature probabilities**. It does not download mail, fit a model, or classify real messages. Run top to bottom with the **AI** kernel; standard Python is sufficient.

## What Bayes' theorem says

`P(A | B) = P(B | A) × P(A) / P(B)`. Say **probability of A given B equals probability of B given A, times probability of A, divided by probability of B**. `A` is a proposed class such as Spam; `B` is observed evidence such as features F₁ and F₂ (say **F sub one** and **F sub two**). `P(A)` is the prior, `P(B | A)` the likelihood, `P(B)` the overall evidence probability, and `P(A | B)` the posterior. In ordinary English: update the starting class rate using how characteristic the observed evidence is, then normalize across possible classes.

In [ ]:
# HTB gives these probabilities; no training data is estimated here.
prior_spam = 0.30
prior_not_spam = 0.70
f1_given_spam, f2_given_spam = 0.40, 0.50
f1_given_not_spam, f2_given_not_spam = 0.20, 0.30
assert abs(prior_spam + prior_not_spam - 1.0) < 1e-12
assert all(0 <= p <= 1 for p in (prior_spam, prior_not_spam,
                                    f1_given_spam, f2_given_spam,
                                    f1_given_not_spam, f2_given_not_spam))

## The *naive* feature assumption

`P(F₁,F₂ | Spam) ≈ P(F₁ | Spam) × P(F₂ | Spam)`. Read **probability of F sub one and F sub two given spam is approximately the probability of F sub one given spam times the probability of F sub two given spam**. This is conditional independence *within* a class, not a general claim that email words are unrelated. Apply the same assumption to Not Spam. The two resulting likelihoods answer *how common these features are within each class*; neither is yet the probability that this particular message is spam.

In [ ]:
# Formula: P(F1,F2 | class) ≈ P(F1 | class) × P(F2 | class).
likelihood_spam = f1_given_spam * f2_given_spam       # 0.40 × 0.50 = 0.20
likelihood_not_spam = f1_given_not_spam * f2_given_not_spam  # 0.20 × 0.30 = 0.06
print('P(features | Spam):', likelihood_spam)
print('P(features | Not Spam):', likelihood_not_spam)
assert abs(likelihood_spam - 0.20) < 1e-12
assert abs(likelihood_not_spam - 0.06) < 1e-12

## Weight by priors and normalize

The **spam path** is `P(features | Spam) × P(Spam) = 0.20 × 0.30 = 0.060`. The **not-spam path** is `0.06 × 0.70 = 0.042`. The law of total probability adds these mutually exclusive paths: `P(features) = 0.060 + 0.042 = 0.102`. Say **probability of features equals the spam path plus the not-spam path**. Dividing each path by 0.102 turns the two unnormalized scores into posterior probabilities that add to one. Never describe the raw 0.060 score as a 6% posterior.

In [ ]:
# Numerators of Bayes' theorem: likelihood × prior for each class.
spam_path = likelihood_spam * prior_spam
not_spam_path = likelihood_not_spam * prior_not_spam
evidence = spam_path + not_spam_path  # P(features), the common denominator.
assert evidence > 0
posterior_spam = spam_path / evidence
posterior_not_spam = not_spam_path / evidence
prediction = 'Spam' if posterior_spam > posterior_not_spam else 'Not Spam'
print(f'P(Spam | features) = {posterior_spam:.3f}')
print(f'P(Not Spam | features) = {posterior_not_spam:.3f}')
print('Predicted class:', prediction)
assert abs(evidence - 0.102) < 1e-12
assert abs(posterior_spam + posterior_not_spam - 1.0) < 1e-12
assert 0.587 < posterior_spam < 0.589 and prediction == 'Spam'

The calculation matches HTB: about **58.8% Spam** versus **41.2% Not Spam** *under the supplied priors and naive assumption*. A higher posterior wins in this classroom example. Real filtering needs trained likelihoods, a representative prior, handling for unseen terms, calibration and a decision threshold chosen for the costs of false alarms versus missed phishing. None of those production steps has been smuggled into this section's notebook.